<div style="width: 100%; clear: both;">
<div style="float: left; width: 50%;">
<img src="https://campusvirtual.urv.cat/pluginfile.php/1/core_admin/logocompact/300x300/1767733831/logoURVppd.png", align="left">
</div>
<div style="float: right; width: 50%;">
<p style="margin: 0; padding-top: 22px; text-align:right;">2025-2026 SCIENTIFIC PROGRAMMING (17715101)</p>
<p style="margin: 0; text-align:right;">MD Health Data Science / Biomedical Data Science</p>
</div>
</div>
<div style="width:100%;">&nbsp;</div>

# Step 5: Model Validation and Selection

**Objective**: Implement validation strategy and select best model for API implementation

**Inputs**: Training and test sets

**Outputs**: Validation metrics, selected model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from pathlib import Path

sys.path.append('../src')

from model.validation import (
    train_test_validation,
    cross_validation,
    select_best_model,
    save_validation_results,
    save_best_model
)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

## 1. Load Data

In [ ]:
df = pd.read_csv('../data/breast_cancer_reduced.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

In [ ]:
y = df['diagnosis'].astype(int)
X = df.drop(['diagnosis', 'diagnosis_label'], axis=1)

print(f"Features: {X.shape[1]} columns")
print(f"Samples: {X.shape[0]}")
print(f"\nTarget distribution:")
print(y.value_counts())
print(f"\nClass balance: {y.value_counts(normalize=True)}")

## 2. Train/Test Split Validation

We'll split the data into 80% training and 20% test sets, then evaluate KNN models with different k values.

In [ ]:
n_neighbors_list = [3, 5, 7, 9, 11, 13, 15]

train_test_results, (X_train, X_test, y_train, y_test) = train_test_validation(
    X, y,
    test_size=0.2,
    random_state=42,
    n_neighbors_list=n_neighbors_list
)

### Visualize Train/Test Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Train/Test Split Validation Results', fontsize=16, fontweight='bold')

metrics = ['accuracy', 'precision', 'recall', 'f1_score']
k_values = []
train_scores = {m: [] for m in metrics}
test_scores = {m: [] for m in metrics}

for model_name, model_data in train_test_results['models'].items():
    k_values.append(model_data['n_neighbors'])
    for metric in metrics:
        train_scores[metric].append(model_data['train_metrics'][metric])
        test_scores[metric].append(model_data['test_metrics'][metric])

for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    ax.plot(k_values, train_scores[metric], marker='o', label='Train', linewidth=2)
    ax.plot(k_values, test_scores[metric], marker='s', label='Test', linewidth=2)
    ax.set_xlabel('Number of Neighbors (k)', fontsize=12)
    ax.set_ylabel(metric.replace('_', ' ').title(), fontsize=12)
    ax.set_title(f'{metric.replace("_", " ").title()} vs k', fontsize=13, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(k_values)

plt.tight_layout()
plt.show()

## 3. Cross-Validation

Perform 5-fold cross-validation to get a more robust estimate of model performance.

In [ ]:
cv_results = cross_validation(
    X, y,
    cv_folds=5,
    random_state=42,
    n_neighbors_list=n_neighbors_list
)

: 

### Visualize Cross-Validation Results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Cross-Validation Results (5-fold)', fontsize=16, fontweight='bold')

metrics_cv = ['accuracy', 'precision', 'recall', 'f1']
k_values_cv = []
cv_means = {m: [] for m in metrics_cv}
cv_stds = {m: [] for m in metrics_cv}

for model_name, model_data in cv_results['models'].items():
    k_values_cv.append(model_data['n_neighbors'])
    for metric in metrics_cv:
        cv_means[metric].append(model_data['cv_metrics'][f'{metric}_mean'])
        cv_stds[metric].append(model_data['cv_metrics'][f'{metric}_std'])

for idx, metric in enumerate(metrics_cv):
    ax = axes[idx // 2, idx % 2]
    means = cv_means[metric]
    stds = cv_stds[metric]
    
    ax.errorbar(k_values_cv, means, yerr=stds, marker='o', capsize=5, 
                linewidth=2, markersize=8, label=f'{metric.title()}')
    ax.fill_between(k_values_cv, 
                     [m - s for m, s in zip(means, stds)],
                     [m + s for m, s in zip(means, stds)],
                     alpha=0.2)
    ax.set_xlabel('Number of Neighbors (k)', fontsize=12)
    ax.set_ylabel(f'{metric.title()} Score', fontsize=12)
    ax.set_title(f'{metric.title()} vs k (with std)', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_xticks(k_values_cv)

plt.tight_layout()
plt.show()

## 4. Model Comparison

Compare all models side by side.

In [ ]:
comparison_data = []

for model_name in train_test_results['models'].keys():
    k = train_test_results['models'][model_name]['n_neighbors']
    
    test_acc = train_test_results['models'][model_name]['test_metrics']['accuracy']
    test_f1 = train_test_results['models'][model_name]['test_metrics']['f1_score']
    
    cv_acc = cv_results['models'][model_name]['cv_metrics']['accuracy_mean']
    cv_acc_std = cv_results['models'][model_name]['cv_metrics']['accuracy_std']
    cv_f1 = cv_results['models'][model_name]['cv_metrics']['f1_mean']
    cv_f1_std = cv_results['models'][model_name]['cv_metrics']['f1_std']
    
    comparison_data.append({
        'k': k,
        'Test Accuracy': f"{test_acc:.4f}",
        'Test F1': f"{test_f1:.4f}",
        'CV Accuracy': f"{cv_acc:.4f} ± {cv_acc_std:.4f}",
        'CV F1': f"{cv_f1:.4f} ± {cv_f1_std:.4f}"
    })

comparison_df = pd.DataFrame(comparison_data)
print("\nModel Comparison Summary:")
print("="*80)
comparison_df

## 5. Model Selection

Select the best model based on validation results.

In [ ]:
best_model_name, selection_summary = select_best_model(
    train_test_results,
    cv_results,
    metric='f1_score'
)

## 6. Best Model Details

In [ ]:
best_model = train_test_results['models'][best_model_name]['model']
best_k = train_test_results['models'][best_model_name]['n_neighbors']

print(f"\n{'='*80}")
print("SELECTED MODEL FOR API IMPLEMENTATION")
print(f"{'='*80}\n")
print(f"Model: KNeighborsClassifier")
print(f"Hyperparameter: n_neighbors = {best_k}")

print(f"\nTest Set Performance:")
test_metrics = train_test_results['models'][best_model_name]['test_metrics']
for metric_name, value in test_metrics.items():
    if metric_name != 'confusion_matrix':
        print(f"  {metric_name}: {value:.4f}")

print(f"\nCross-Validation Performance:")
cv_metrics = cv_results['models'][best_model_name]['cv_metrics']
print(f"  Accuracy: {cv_metrics['accuracy_mean']:.4f} (+/- {cv_metrics['accuracy_std']:.4f})")
print(f"  Precision: {cv_metrics['precision_mean']:.4f} (+/- {cv_metrics['precision_std']:.4f})")
print(f"  Recall: {cv_metrics['recall_mean']:.4f} (+/- {cv_metrics['recall_std']:.4f})")
print(f"  F1-Score: {cv_metrics['f1_mean']:.4f} (+/- {cv_metrics['f1_std']:.4f})")

### Confusion Matrix for Best Model

In [ ]:
cm = np.array(test_metrics['confusion_matrix'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Benign (0)', 'Malignant (1)'],
            yticklabels=['Benign (0)', 'Malignant (1)'])
plt.title(f'Confusion Matrix - Best Model (k={best_k})', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"\nConfusion Matrix Breakdown:")
print(f"  True Negatives (TN): {tn}")
print(f"  False Positives (FP): {fp}")
print(f"  False Negatives (FN): {fn}")
print(f"  True Positives (TP): {tp}")

## 7. Save Results

In [ ]:
results_path = Path('../results')
results_path.mkdir(exist_ok=True)

save_validation_results(
    train_test_results,
    str(results_path / 'train_test_validation_results.json')
)

save_validation_results(
    cv_results,
    str(results_path / 'cross_validation_results.json')
)

save_validation_results(
    selection_summary,
    str(results_path / 'model_selection_summary.json')
)

save_best_model(
    best_model,
    str(results_path / 'best_model.pkl')
)

print("\n✓ All results saved successfully!")